In [ ]:
!pip install web3

In [2]:
from web3 import Web3
from datetime import datetime

# ────────────────────────────────────────────────
# USDC (ERC-20) Contract Info – Ethereum Mainnet (as of 2026)
# ────────────────────────────────────────────────
USDC_ADDRESS = "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"
USDC_DECIMALS = 6  # USDC has 6 decimals (unlike ETH's 18)

# Minimal ERC-20 ABI – only what we need for balanceOf
ERC20_ABI = [
    {
        "constant": True,
        "inputs": [{"name": "_owner", "type": "address"}],
        "name": "balanceOf",
        "outputs": [{"name": "balance", "type": "uint256"}],
        "type": "function"
    },
    {
        "constant": True,
        "inputs": [],
        "name": "decimals",
        "outputs": [{"name": "", "type": "uint8"}],
        "type": "function"
    }
]

# Free public RPC endpoints (try in order – they rotate)
RPC_URLS = [
    "https://ethereum-rpc.publicnode.com",
    "https://rpc.ankr.com/eth",
    "https://eth.llamarpc.com",
]

# ────────────────────────────────────────────────
# Connect to Ethereum
# ────────────────────────────────────────────────
w3 = None
for url in RPC_URLS:
    try:
        temp_w3 = Web3(Web3.HTTPProvider(url))
        if temp_w3.is_connected():
            w3 = temp_w3
            print(f"Connected to Ethereum mainnet via: {url}")
            break
    except Exception as e:
        print(f"Failed {url}: {e}")

if w3 is None:
    print("No public RPC available right now. Try later or add your own Infura/Alchemy key.")
    exit(1)

# ────────────────────────────────────────────────
# Load USDC contract
# ────────────────────────────────────────────────
usdc_contract = w3.eth.contract(address=USDC_ADDRESS, abi=ERC20_ABI)

# ────────────────────────────────────────────────
# Functions
# ────────────────────────────────────────────────
def get_usdc_balance(address: str):
    """Check USDC balance for any Ethereum address"""
    try:
        # Convert to checksum format (best practice)
        checksum_addr = w3.to_checksum_address(address)
        
        # Call balanceOf
        balance_raw = usdc_contract.functions.balanceOf(checksum_addr).call()
        
        # Convert from smallest unit (6 decimals)
        balance_usdc = balance_raw / (10 ** USDC_DECIMALS)
        
        print(f"\nUSDC Balance Check ({datetime.now().strftime('%Y-%m-%d %H:%M UTC')})")
        print("═══════════════════════════════════════════")
        print(f"Address           : {checksum_addr}")
        print(f"USDC Balance      : {balance_usdc:,.6f} USDC")
        print(f"Raw balance       : {balance_raw:,} (wei-like units)")
        print("═══════════════════════════════════════════")
        
        # Optional: Confirm decimals from chain (should be 6)
        chain_decimals = usdc_contract.functions.decimals().call()
        print(f"Confirmed decimals: {chain_decimals}")
        
    except Exception as e:
        print(f"Error: {e}")
        print("Possible causes: invalid address, RPC issue, or contract call failed.")

# ────────────────────────────────────────────────
# Run the demo
# ────────────────────────────────────────────────
if __name__ == "__main__":
    print("Ethereum USDC Stablecoin Balance Demo\n")
    
    # Example: a public/hot address often used in demos (replace with yours!)
    example_address = "0xAb5801a7D398351b8bE11C439e05C5B3259aeC9B"  # Vitalik's known address (small USDC usually)
    
    print(f"Checking example address: {example_address}")
    get_usdc_balance(example_address)
    
    # Uncomment and replace to check your own wallet
    # your_address = "0xYourEthereumAddressHere"
    # get_usdc_balance(your_address)
    
    print("\nTip: Paste any Ethereum address above to check its USDC holdings.")
    print("Want to extend? We can add transfers, USDT/DAI, or allowance checks next!")

Connected to Ethereum mainnet via: https://ethereum-rpc.publicnode.com
Ethereum USDC Stablecoin Balance Demo

Checking example address: 0xAb5801a7D398351b8bE11C439e05C5B3259aeC9B

USDC Balance Check (2026-02-10 00:54 UTC)
═══════════════════════════════════════════
Address           : 0xAb5801a7D398351b8bE11C439e05C5B3259aeC9B
USDC Balance      : 0.000000 USDC
Raw balance       : 0 (wei-like units)
═══════════════════════════════════════════
Confirmed decimals: 6

Tip: Paste any Ethereum address above to check its USDC holdings.
Want to extend? We can add transfers, USDT/DAI, or allowance checks next!
